# ReplicaLab Minimal Colab Training Script

This is the sponsor-facing minimal training notebook for ReplicaLab.

It uses **Unsloth + HF TRL GRPO** in Colab to train the Scientist policy used by the ReplicaLab environment.

What this notebook does:
1. install the repo and the Unsloth / TRL stack
2. load the frozen evidence packs used by ReplicaLab training
3. preview a tiny Scientist GRPO config
4. optionally run one minimal GRPO step with the Colab-friendly `Qwen/Qwen3-4B` fallback
5. inspect the saved artifacts

Use `notebooks/train_colab.ipynb` for the full V2 notebook with the broader Scientist, Lab Manager, evaluation, and infra workflow.


In [ ]:
%%capture
!pip install --upgrade -qqq uv
!uv pip install -e .
!uv pip install unsloth unsloth_zoo trl vllm datasets matplotlib


In [ ]:
import json

from replicalab.training import (
    ArtifactLayout,
    ScientistGRPOConfig,
    load_frozen_evidence_packs,
    preview_scientist_training,
    train_scientist_grpo,
)

RUN_REAL_TRAINING = False
MODEL_NAME = "Qwen/Qwen3-4B"
layout = ArtifactLayout.create(run_name="minimal-colab-scientist")
evidence_packs = load_frozen_evidence_packs()
print(f"Loaded {len(evidence_packs)} evidence packs")


In [ ]:
config = ScientistGRPOConfig(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    fast_inference=True,
    train_seeds=[0],
    templates=["ml_benchmark"],
    difficulties=["easy"],
    max_steps=1,
    num_generations=2,
    max_completion_length=256,
    logging_steps=1,
    save_steps=1,
)

plan = preview_scientist_training(config, layout=layout)
plan


In [ ]:
if RUN_REAL_TRAINING:
    result = train_scientist_grpo(config, layout=layout)
    print(json.dumps(result, indent=2))
else:
    print("Preview only. Set RUN_REAL_TRAINING = True to execute one minimal GRPO step in Colab.")


In [ ]:
print("Run directory:", layout.run_dir)
print("Config file exists:", layout.config_json.exists())
print("Evidence manifest exists:", layout.evidence_manifest_json.exists())
print("Adapter directory exists:", layout.scientist_adapter_dir.exists())

if layout.config_json.exists():
    print(layout.config_json.read_text(encoding="utf-8")[:1000])


## Notes

- This notebook is the minimal Colab artifact for the hackathon requirement.
- It trains against the same ReplicaLab prompt, evidence-pack, and deterministic reward stack used by the main environment.
- For the full judged flow, use `notebooks/train_colab.ipynb`.
- For live OpenEnv rollouts against the hosted environment, use the ART/OpenEnv CLI path documented in `docs/ayush/notebook_smoke_test.md`.
